In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [3]:
df = pd.read_csv("../datasets/ISOT/isot.csv")
df

,article,label
0,Heaven’s Gatekeeper? Jesse Jackson Proclaims T...,0
1,Obama And Justin Trudeau Had Dinner Tuesday; ...,0
2,"U.S. willing, if asked, to facilitate talks be...",1
3,Elysee plays down opulence of Macron birthday ...,1
4,CNN Anchor Asks Van Jones To Take Back His Pra...,0
...,...,...
9995,FATHER OF STUDENT Released From N. Korean Pris...,0
9996,‘We Know Where You Live’: Trump-Loving Terror...,0
9997,New Russian ambassador to U.S. calls for resum...,1
9998,FOUR CANDIDATES FOR FBI DIRECTOR Interviewed T...,0


In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["article"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=10,
    save_steps=100,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\nikita.bardatskii\miniforge3\envs\bi-bap\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.000600,0.017295
2,0.000100,0.007694
3,0.000000,0.010644


TrainOutput(global_step=3000, training_loss=0.030630216048443498, metrics={'train_runtime': 1497.7757, 'train_samples_per_second': 16.024, 'train_steps_per_second': 2.003, 'total_flos': 3157332664320000.0, 'train_loss': 0.030630216048443498, 'epoch': 3.0})

In [7]:
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [8]:
# Assuming you already have your trained model and test_dataset

# Define your compute_metrics function if you haven't already
import numpy as np
from scipy.special import softmax
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)[:, 1]
    accuracy = accuracy_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    roc_auc = roc_auc_score(labels, probs)
    return {"accuracy": accuracy, "recall": recall, "f1": f1, "roc_auc": roc_auc}

# Reinitialize or update your Trainer with compute_metrics (no training is triggered here)
from transformers import Trainer

trainer = Trainer(
    model=model,                # Your trained model
    eval_dataset=test_dataset,  # Evaluation dataset
    compute_metrics=compute_metrics
)

# Evaluate the model without retraining
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.010644100606441498, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.9985, 'eval_recall': 1.0, 'eval_f1': 0.9985815602836879, 'eval_roc_auc': 0.9999989968541345, 'eval_runtime': 32.7211, 'eval_samples_per_second': 61.123, 'eval_steps_per_second': 7.64}
